# Training model with distributed learning pipeline (experimental)


## Verification of dependencies and their versions

In [1]:
import time
from datetime import datetime

import kubeflow
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print("All imports successful!")

Kubeflow version: 0.3.0
All imports successful!


## PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes. `TrainerClient()` verifies that you have required access to the Kubernetes cluster. Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [2]:

from kubeflow.trainer import CustomTrainer, TrainerClient
config = kubeflow.trainer.KubernetesBackendConfig()

client = TrainerClient()
trainer = TrainerClient()
# trainer = TrainerClient(backend_config=config) #TOTEST

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob. Additionally, it might show available accelerator type and number of available resources.

In [3]:
for runtime in trainer.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='retfound-image-generation', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/yukun-image-generation-runner:v0.0.8', num_nodes=1, device='gpu', device_count='1'), pretrained_model=None)
Runtime(name='retfound-pretraining', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='ghcr.io/andesterson/dinov3_gr_pretrain-ru

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [4]:
from kubeflow.trainer import TrainerClient, CustomTrainer, options
from kubeflow.trainer.options import TrainerCommand
from kubeflow.trainer.types.types import CustomTrainerContainer

# ------------------------------------------------
# Configuration of resources
# ------------------------------------------------

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 1

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "3",           # CPUs per node
    "memory": "2Gi",     # Memory in GiB per node
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}


### AVAILABLE CONTAINER IMAGES

# fetal-ultrasound-edm2-distributed-learning:v0.0.5
# docker.io/pytorch/pytorch:2.7.1-cuda12.8-cudnn9-devel

# fetal-ultrasound-edm2-distributed-learning:v0.0.6
# docker.io/pytorch/pytorch:2.9.1-cuda12.8-cudnn9-devel


## Set github container registry
# GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.0.5"
GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.0.6"

# ------------------------------------------------
# Seeting up to mount scratch-volume
# ------------------------------------------------
volume_name = "scratch-volume"
pvc_name = "scratch-volume"
mount_scrath_path = "/scratch-volume"

pod_volumes = [
    {
        "name": volume_name,
        "persistentVolumeClaim": {
            "claimName": pvc_name
        }
    }
]
volume_mounts = [
    {
        "name": volume_name,
        "mountPath": mount_scrath_path
    }
]
container_override = options.ContainerOverride(
    name="node",
    volume_mounts=volume_mounts
)
pod_spec_override = options.PodSpecOverride(
    volumes=pod_volumes,
    containers=[container_override]
)
pod_template_override = options.PodTemplateOverride(
    target_jobs=["node"],
    spec=pod_spec_override
)
pod_template_overrides = options.PodTemplateOverrides(
    pod_template_override
)


ENV_VARS = {
    "MASTER_ADDR": "localhost",
    "MASTER_PORT": "12355",
    "WORLD_SIZE": "1",
    "RANK": "0",
    "LOCAL_RANK": "0",
}


#
#--standalone": 
# NUM_NODES = 1 set up at the begining of the cell
NPROC_PER_NODE=1

command = TrainerCommand(
    command=[
        "torchrun",
        # "--standalone", #Single-node multi-worker # pragma: allowlist secret
        # f"--nnodes={NUM_NODES}",
        # f"--nproc_per_node={NPROC_PER_NODE}",
        # "--node_rank={NODE_RANK}",
        # f"--master_addr={MASTER_ADDR}",
        # f"--master_port={MASTER_PORT}",
        "train_edm2.py",
        "--outdir", "/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY", # pragma: allowlist secret
        "--data", "/scratch-volume/FETAL_PLANES_DB", # pragma: allowlist secret
        "--batch", "8",
        "--preset", "edm2-img512-s",
        "--batch-gpu", "8",
    ]
)





In [5]:
job_id = trainer.train(
    runtime=torch_runtime,
    trainer=CustomTrainerContainer(
        image=GITHUB_CONTAINER_REGISTRY,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
        env=ENV_VARS        
    ),
    options=[command, pod_template_overrides],
)


In [6]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: udfa84d1008f
Job Status: Created
Creation Time: 2026-04-15 10:21:04+00:00

Job details: TrainJob(name='udfa84d1008f', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=1, creation_timestamp=datetime.datetime(2026, 4, 15, 10, 21, 4, tzinfo=TzInfo(0)), status='Created')


In [7]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


### EXPECTED TERMINAL LOG FROM LOCAL MACHINE:
#### COMMAND
# torchrun  train_edm2.py 
# --outdir ~/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY 
# --data ~/scratch-volume/FETAL_PLANES_DB             
# --batch 8             
# --preset edm2-img512-s            
# --batch-gpu=8

#### LOGS
# Output directory:        /home/mxochicale/scratch-volume/FETAL_PLANES_DB/OUTPUT_DIRECTORY
# Dataset path:            /home/mxochicale/scratch-volume/FETAL_PLANES_DB
# Class-conditional:       True
# Number of GPUs:          1
# Batch size:              8
# Mixed-precision:         True

# Loading dataset...
# Dataset size: 7129
# Setting up encoder...
# Constructing network...
# {'class_name': 'training.networks_edm2.Precond', 'model_channels': 192, 'dropout': 0.0, 'use_fp16': True} {'img_resolution': 64, 'img_channels': 4, 'label_dim': 6}

# Precond                Parameters  Buffers  Output shape      Datatype
# ---                    ---         ---      ---               ---     
# unet.emb_fourier       -           384      [8, 192]          float32 
# unet.emb_noise         147456      -        [8, 768]          float32 
# unet.emb_label         4608        -        [8, 768]          float32 
# ...
# unet.dec.64x64_block3  1216513     -        [8, 192, 64, 64]  float16 
# unet.out_conv          6912        -        [8, 4, 64, 64]    float16 
# unet                   1           -        [8, 4, 64, 64]    float16 
# <top-level>            128         256      [8, 4, 64, 64]    float32 
# ---                    ---         ---      ---               ---     
# Total                  279449445   640      -                 -       

# Setting up training state...
# Training from 0 kimg to 2147483 kimg:

# Setting up validation set...



Waiting for job logs...
[10:21:04] Waiting... (1s)
[10:21:05] Waiting... (2s)
[10:21:07] Waiting... (3s)
Logs received after 3 seconds:
  Setting up training config...
  Training config:
  {
    "dataset_kwargs": {
      "class_name": "training.dataset.UltrasoundDataset",
      "path": "/scratch-volume/FETAL_PLANES_DB",
      "use_labels": true,
      "fpus23_path": null,
      "african_path": null,
      "fetal_abdomen_path": null
    },
    "val_kwargs": {
      "class_name": "training.dataset.UltrasoundDataset",
      "path": "/scratch-volume/FETAL_PLANES_DB",
      "use_labels": true,
      "split": "val",
      "num_classes": 6
    },
    "encoder_kwargs": {
      "class_name": "training.encoders.StabilityVAEEncoder"
    },
    "total_nimg": 2147483648,
    "batch_size": 8,
    "network_kwargs": {
      "class_name": "training.networks_edm2.Precond",
      "model_channels": 192,
      "dropout": 0.0,
      "use_fp16": true
    },
    "loss_kwargs": {
      "class_name": "training.

In [8]:
# Delete the TrainJob
# When TrainJob is finished, you can delete the resource.
# client.delete_job(job_id)